# 02 — Book-level split
Keep every page, text, body, face, and frame in the joined records. Only explicit, unambiguous visible-speaker links supply training labels. Unlinked text stays `UNLABELED`. Boxes remain original-image pixel XYXY. Distances can be computed from these retained boxes for every text/character combination.


In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi\Manga109Association")
sys.path.insert(0, str(ROOT))
from common import *

DATA.mkdir(exist_ok=True)
CHECKPOINTS.mkdir(exist_ok=True)


In [ ]:
import random

settings = read_json(DATA / "dataset_settings.json")
pages, speaker_types, issues = load_pages(settings["base_xml_dir"], settings["dialog_xml_dir"], settings["image_dir"])
inspection = read_json(DATA / "inspection_ok.json")
assert inspection["joined_hash"] == digest(pages), "Annotations changed: repeat notebook 01."
assert inspection["dataset_settings_hash"] == digest(settings)
assert len(inspection["pages_reviewed"]) >= 30

VAL_FRACTION = 0.20
books = sorted({p["book"] for p in pages})
assert len(books) >= 2
random.Random(SEED).shuffle(books)
val_count = min(len(books) - 1, max(1, round(len(books) * VAL_FRACTION)))
val_books = set(books[:val_count])
train_pages = [p for p in pages if p["book"] not in val_books]
val_pages = [p for p in pages if p["book"] in val_books]
assert {p["book"] for p in train_pages}.isdisjoint({p["book"] for p in val_pages})
assert sum(len(p["speaker_links"]) for p in train_pages) > 0
assert sum(len(p["speaker_links"]) for p in val_pages) > 0

save_json(DATA / "train_pages.json", train_pages)
save_json(DATA / "val_pages.json", val_pages)
save_json(DATA / "split.json", {"seed": SEED, "val_fraction": VAL_FRACTION, "train_books": sorted(set(books) - val_books), "val_books": sorted(val_books)})
for name, records in [("train", train_pages), ("val", val_pages)]:
    print(name, "books:", len({p["book"] for p in records}), "pages:", len(records), "linked texts:", sum(len(p["speaker_links"]) for p in records))
